# 01 — Physics Concept Extraction & Validation  ·  Critical Fix (M35 → concepts) + Gate G2 + CC2

**Current project direction:** physics-grounded, *label-free* acoustic **concept bottleneck** + faithfulness audit.
This notebook is the **first Critical Fix**: it turns the M35 physics DSP into standalone, clinically-named
**per-cycle concept extractors**, runs them on ICBHI (official 60/40 split), and **validates them against ICBHI's
crackle/wheeze cycle labels (Gate G2)**. It also runs the cheap **device-structure check (CC2)**.

**What "pass" looks like (G2):** the continuous `crackle_presence` / `wheeze_presence` concepts should track the
ICBHI labels with AUROC clearly above 0.5. If they do, the physics concepts are clinically meaningful and you may
build the bottleneck (notebook 02). If not, fall back to the minimal reliable concept set.

Outputs: `concepts_all.npz` (concept vectors + metadata), `concept_validation_report.json`, device report.


In [ ]:
# --- setup: make the owmtl package importable ---------------------------------
# On Kaggle: upload the `owmtl/` folder as a dataset and point OWMTL_PKG at it,
# or place owmtl/ next to this notebook.
import sys, os
OWMTL_PKG = ".."          # TODO: path that CONTAINS the `owmtl` folder
sys.path.insert(0, OWMTL_PKG)

import numpy as np, json, time
from owmtl import icbhi_data as D
from owmtl.concept_extractors import (extract_concept_vector, CONCEPT_NAMES,
                                       CONCEPT_LABEL_MAP, ConceptConfig)
from owmtl import eval_utils as E
print("concepts:", CONCEPT_NAMES)


In [ ]:
# --- CONFIG: point these at the real ICBHI files -----------------------------
AUDIO_DIR = "/kaggle/input/icbhi-dataset/audio_and_txt_files"   # TODO
SPLIT_FILE = "/kaggle/input/icbhi-dataset/ICBHI_challenge_train_test.txt"  # TODO official 60/40
DIAG_FILE  = "/kaggle/input/icbhi-dataset/ICBHI_Challenge_diagnosis.txt"   # TODO
SR = 16000
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)


### CC2 — device-structure feasibility (Gate G4). Cheap; decides the device axis now.

In [ ]:
from owmtl.device_check import analyze, format_report
dev_report = analyze(AUDIO_DIR, DIAG_FILE)
print(format_report(dev_report))
with open(os.path.join(OUT_DIR, "device_structure_report.json"), "w") as fh:
    json.dump(dev_report, fh, indent=2)


### Build the cycle index on the OFFICIAL split (patient-independent by construction).

In [ ]:
records = D.build_cycle_index(AUDIO_DIR, SPLIT_FILE, DIAG_FILE)
from collections import Counter
print("cycles:", len(records))
print("split :", Counter(r.split for r in records))
print("sound :", Counter(r.sound_label for r in records), "(0=N,1=C,2=W,3=B)")
print("device:", Counter(r.device for r in records))


### Extract the physics concept vector for every cycle (label-free DSP).

In [ ]:
cfg = ConceptConfig(sr=SR)
X = np.zeros((len(records), len(CONCEPT_NAMES)), dtype=np.float32)
meta = {k: [] for k in ("patient","device","split","crackle","wheeze","sound_label","diagnosis")}
t0 = time.time()
for i, r in enumerate(records):
    try:
        y = D.load_cycle_waveform(AUDIO_DIR, r, sr=SR)
        X[i] = extract_concept_vector(y, SR, cfg)
    except Exception as ex:
        X[i] = 0.0
        if i < 5: print("warn", r.stem, ex)
    for k in meta: meta[k].append(getattr(r, k))
    if (i+1) % 500 == 0: print(f"{i+1}/{len(records)}  ({time.time()-t0:.0f}s)")
meta = {k: np.array(v) for k, v in meta.items()}
np.savez_compressed(os.path.join(OUT_DIR, "concepts_all.npz"),
                    X=X, concept_names=np.array(CONCEPT_NAMES), **meta)
print("saved concepts_all.npz  shape", X.shape)


### Gate G2 — validate concepts against ICBHI labels
`crackle_presence` vs the ICBHI `crackle` bit, `wheeze_presence` vs `wheeze` bit, on the **test** split, with
bootstrap 95% CIs (Protocol Essential #10). AUROC clearly > 0.5 ⇒ the physics concept tracks the clinical label.

In [ ]:
test = meta["split"] == "test"
report = {"gate": "G2", "n_test_cycles": int(test.sum()), "concept_label_auroc": {}}
name_idx = {n: i for i, n in enumerate(CONCEPT_NAMES)}
for concept, labelname in CONCEPT_LABEL_MAP.items():
    y = meta[labelname][test].astype(int)
    s = X[test, name_idx[concept]]
    pt, lo, hi = E.auroc_ci(y, s, n_boot=1000)
    report["concept_label_auroc"][concept] = {"vs_label": labelname,
        "auroc": round(pt,4), "ci95": [round(lo,4), round(hi,4)],
        "n_pos": int(y.sum()), "n_neg": int((1-y).sum())}
    print(f"{concept:18s} vs {labelname:8s}: AUROC {pt:.3f}  95% CI [{lo:.3f},{hi:.3f}]  (n+={y.sum()})")

# also: how well the full concept vector linearly separates each sound label (sanity)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
tr = ~test
for name, lab in [("crackle","crackle"),("wheeze","wheeze")]:
    try:
        clf = LogisticRegression(max_iter=500, class_weight="balanced").fit(X[tr], meta[lab][tr])
        auc = roc_auc_score(meta[lab][test], clf.predict_proba(X[test])[:,1])
        report.setdefault("full_vector_auroc", {})[lab] = round(float(auc),4)
        print(f"full concept vector -> {lab}: AUROC {auc:.3f}")
    except Exception as ex:
        print("skip", lab, ex)

with open(os.path.join(OUT_DIR, "concept_validation_report.json"), "w") as fh:
    json.dump(report, fh, indent=2)


### G2 decision
- **PASS** (single-concept AUROC materially > 0.5, e.g. ≳ 0.6, CI lower bound > 0.5): the physics concepts are
  clinically meaningful → proceed to notebook 02 (bottleneck). The full-vector AUROCs should be higher still.
- **BORDERLINE/FAIL:** shrink to the most reliable concepts (PAPR, spectral_flatness, wheeze band) or add a
  learned concept-refinement step; re-validate before building the bottleneck.

This report (`concept_validation_report.json`) is the evidence a reviewer will ask for that "physics concepts" are real.
